# 01 - Inventário dos Dados Brutos e Auditoria do Dicionário
**Projeto:** Vitaliza (Módulo 2 MBA Inteli)  
**Autor:** Antigravity AI  
**Data:** 22/10/2026  
**Ambiente:** `dev`  

---
## Objetivo
Executar a auditoria e inventário completo dos 12 arquivos tabulares e dos pacotes de mapas de calor contidos em `dados/brutos/`, comparando metadados reais (linhas, colunas, tipos de dados e valores vazios) com a especificação formal em `docs/Vitaliza - Dicionário de dados.pdf`.


In [1]:
import os
import zipfile
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

RAW_DIR = "../dados/brutos"
print(f"Diretório de dados brutos: {os.path.abspath(RAW_DIR)}")


Diretório de dados brutos: C:\Projetos\Vitaliza\dados\brutos


## 1. Dicionário de Especificações Esperadas
Definição dos metadados extraídos formalmente do `Vitaliza: dicionário de dados`.


In [2]:
expected_dict = {
    "usuarios.csv": {
        "linhas": 64948,
        "colunas": ["user_id", "email", "data_cadastro", "data_nascimento", "sexo", "cidade", "uf", "cep_prefixo", "canal_aquisicao", "sistema", "versao_app", "relogio_conectado"]
    },
    "assinaturas.csv": {
        "linhas": 32643,
        "colunas": ["assinatura_id", "user_id", "plano_atual", "canal_cobranca", "inicio", "status_atual", "renovacao_automatica", "data_fim", "motivo_fim"]
    },
    "amizades.csv": {
        "linhas": 113480,
        "colunas": ["seguidor_id", "seguido_id", "desde"]
    },
    "campanha_ago26.csv": {
        "linhas": 14120,
        "colunas": ["user_id", "grupo", "enviado_em", "canal", "titulo_mensagem", "oferta", "abriu_push", "abriu_email", "clicou"]
    },
    "eventos_assinatura.csv": {
        "linhas": 275565,
        "colunas": ["user_id", "data_hora", "evento", "plano", "canal_cobranca", "detalhe"]
    },
    "pagamentos.csv": {
        "linhas": 165548,
        "colunas": ["pagamento_id", "user_id", "data_hora", "valor", "metodo", "status", "tentativa"]
    },
    "uso_mensal.csv": {
        "linhas": 591662,
        "colunas": ["user_id", "mes", "atividades", "atividades_ar_livre", "atividades_video", "minutos", "dias_ativos", "sessoes_app"]
    },
    "atividades.parquet": {
        "linhas": 1254475,
        "colunas": ["atividade_id", "user_id", "inicio", "tipo", "duracao_min", "distancia_km", "ritmo_min_km", "fc_media", "origem", "gps_ok", "treino_com"]
    },
    "onboarding.csv": {
        "linhas": 63929,
        "colunas": ["onboarding_id", "user_id", "respondido_em", "objetivo_texto", "como_conheceu", "nivel", "dias_por_semana"]
    },
    "pesquisa_saida.csv": {
        "linhas": 643,
        "colunas": ["resposta_id", "user_id", "respondido_em", "motivo_principal", "comentario"]
    },
    "feed_posts.csv": {
        "linhas": 119309,
        "colunas": ["post_id", "user_id", "data_hora", "texto", "curtidas"]
    },
    "tickets_suporte.csv": {
        "linhas": 4422,
        "colunas": ["ticket_id", "user_id", "aberto_em", "canal", "texto_cliente", "categoria_atendente", "status", "resolvido_em"]
    }
}


## 2. Execução do Inventário e Perfilamento dos Arquivos Tabulares
Cálculo de contagem exata de linhas, conformidade de colunas, dtypes e quantificação de valores ausentes/vazios.


In [3]:
summary_rows = []
detailed_profiles = {}

for filename, spec in expected_dict.items():
    filepath = os.path.join(RAW_DIR, filename)
    if filename.endswith(".csv"):
        df = pd.read_csv(filepath, low_memory=False)
    else:
        df = pd.read_parquet(filepath)
        
    n_rows = len(df)
    n_cols = len(df.columns)
    
    colunas_reais = list(df.columns)
    missing_cols = [c for c in spec["colunas"] if c not in colunas_reais]
    extra_cols = [c for c in colunas_reais if c not in spec["colunas"]]
    
    summary_rows.append({
        "Arquivo": filename,
        "Linhas Reais": n_rows,
        "Linhas Esperadas": spec["linhas"],
        "Diff Linhas": n_rows - spec["linhas"],
        "Cols Reais": n_cols,
        "Cols Esperadas": len(spec["colunas"]),
        "Cols Faltando": len(missing_cols),
        "Cols Extras": len(extra_cols)
    })
    
    # Detalhamento de colunas
    col_stats = []
    for col in df.columns:
        s = df[col]
        n_null = int(s.isna().sum())
        if s.dtype == 'object' or str(s.dtype) == 'string':
            s_str = s.fillna('').astype(str).str.strip()
            n_blank = int((s_str == '').sum()) - n_null
            if n_blank < 0: n_blank = 0
        else:
            n_blank = 0
        total_missing = n_null + n_blank
        col_stats.append({
            "Coluna": col,
            "Tipo": str(s.dtype),
            "NaNs": n_null,
            "Strings Vazias": n_blank,
            "Total Ausente": total_missing,
            "% Ausente": round((total_missing / n_rows) * 100, 2)
        })
    detailed_profiles[filename] = pd.DataFrame(col_stats)

df_summary = pd.DataFrame(summary_rows)
display(df_summary)


,Arquivo,Linhas Reais,Linhas Esperadas,Diff Linhas,Cols Reais,Cols Esperadas,Cols Faltando,Cols Extras
0,usuarios.csv,64948,64948,0,12,12,0,0
1,assinaturas.csv,32643,32643,0,9,9,0,0
2,amizades.csv,113480,113480,0,3,3,0,0
3,campanha_ago26.csv,14120,14120,0,9,9,0,0
4,eventos_assinatura.csv,275565,275565,0,6,6,0,0
5,pagamentos.csv,165548,165548,0,7,7,0,0
6,uso_mensal.csv,591662,591662,0,8,8,0,0
7,atividades.parquet,1254475,1254475,0,11,11,0,0
8,onboarding.csv,63929,63929,0,7,7,0,0
9,pesquisa_saida.csv,643,643,0,5,5,0,0


## 3. Inventário dos Mapas de Calor (`heatmaps`)
Inspeção dos pacotes `.zip` contendo os heatmaps de 90 dias em 31/07/2026 e 30/09/2026.


In [4]:
heatmaps_summary = []
hdir = os.path.join(RAW_DIR, "heatmaps")

for zfile in sorted(os.listdir(hdir)):
    if zfile.endswith(".zip"):
        zpath = os.path.join(hdir, zfile)
        with zipfile.ZipFile(zpath, 'r') as z:
            pngs = [n for n in z.namelist() if n.lower().endswith(".png")]
            data_str = zfile.replace("heatmaps_", "").replace(".zip", "")
            esp = 12426 if "2026-07-31" in zfile else 13047
            heatmaps_summary.append({
                "Data Corte": data_str,
                "Arquivo": zfile,
                "PNGs Reais": len(pngs),
                "PNGs Esperados": esp,
                "Diferença": len(pngs) - esp,
                "Exemplo": pngs[0] if pngs else None
            })

df_heatmaps = pd.DataFrame(heatmaps_summary)
display(df_heatmaps)


,Data Corte,Arquivo,PNGs Reais,PNGs Esperados,Diferença,Exemplo
0,2026-07-31,heatmaps_2026-07-31.zip,12431,12426,5,2026-07-31/u_0aa0410a.png
1,2026-09-30,heatmaps_2026-09-30.zip,13047,13047,0,2026-09-30/u_0aa0410a.png


## 4. Análise de Inconsistências e Anomalias Específicas


In [5]:
# 4.1 Inconsistência em pagamentos.csv: formato de valores
df_pag = pd.read_csv(os.path.join(RAW_DIR, "pagamentos.csv"), low_memory=False)
mask_rs = df_pag['valor'].astype(str).str.contains('R$')
print(f"Total pagamentos com 'R$': {mask_rs.sum():,} de {len(df_pag):,}")
print("Distribuição por método:")
print(df_pag[mask_rs]['metodo'].value_counts())


Total pagamentos com 'R$': 0 de 165,548
Distribuição por método:
Series([], Name: count, dtype: int64)


In [6]:
# 4.2 Inconsistência em atividades.parquet: treinos solo vs companhia
df_ativ = pd.read_parquet(os.path.join(RAW_DIR, "atividades.parquet"))
n_solo = (df_ativ['treino_com'] == '').sum()
print(f"Treinos solo (string vazia ''): {n_solo:,} ({n_solo/len(df_ativ)*100:.2f}%)")
print(f"Treinos com parceiro: {(df_ativ['treino_com'] != '').sum():,}")


Treinos solo (string vazia ''): 1,240,338 (98.87%)
Treinos com parceiro: 14,137
